# Tuần 2 — Đánh giá Ba Biến thể Prompt P1/P2/P3 & Kiểm tra Top-10 Logits (TV3 — Tasks 3.5 & 3.6)

Notebook này thực hiện **Task 3.5** và **Task 3.6** trong `docs/WEEK2.md` và `docs/guides/TV3-tuan2.md`:
1. **Nạp dữ liệu theo Hợp đồng chung (`train.data.load_split_data`):** Đọc file split dạng ID-only `data/splits/cluster_split.jsonl` (kiểm tra mã băm SHA-256 `8c2c35ea789acf5b2bd9c74c5e59fa0df325034422be42ae7b47f2e8c9c46aaa`), ghép văn bản từ `data/raw/fns/{train,validation}.jsonl` và ánh xạ nhãn về thứ tự chuẩn `LABELS = ("bearish", "neutral", "bullish")` (`0, 1, 2`).
2. **Ba biến thể Prompt (`P1`, `P2`, `P3`):**
   - `P1`: Prompt zero-shot ngắn gọn từ Tuần 1.
   - `P2`: Prompt zero-shot kèm định nghĩa chi tiết của 3 nhãn `bearish`, `bullish`, `neutral` theo `labeling/GUIDE.md`.
   - `P3`: Prompt few-shot kèm 3 ví dụ mẫu (1 ví dụ mỗi nhãn, lấy ngẫu nhiên có `seed = 1` **chỉ từ tập `train`**, tuyệt đối không lấy từ `validation` hay `test`).
3. **Đánh giá trên Validation & Chẩn đoán Top-10 Logits (`evaluate.metrics`):**
   - Chạy `Qwen/Qwen2.5-7B-Instruct` (`nf4-4bit`, `float16`) trên toàn bộ **1.791 mẫu `validation`**.
   - Ghi dự đoán chuẩn vào `results/E11/{P1,P2,P3}/preds_validation.jsonl`.
   - Ghi bảng tổng hợp `results/llm_prompts/top10_check.json` (gồm `prompt`, `n`, `min_abc_mass`, `mean_abc_mass`, `share_top1_is_letter`, `macro_f1`, `f1`, `confusion` — **không chứa văn bản tiêu đề**).

## 1. Cài đặt Môi trường Colab, Clone Nhánh `tv3/week2` & Tải Dữ liệu Thô FNS

- Sử dụng `HF_TOKEN` và `GH_TOKEN` từ **Colab Secrets** (`google.colab.userdata`) với cơ chế HTTP header tạm thời (không lưu token vào `.git/config` hay cell output).
- Chạy `python data/download.py --only fns` để tạo `data/raw/fns/train.jsonl` và `data/raw/fns/validation.jsonl` có ID chuẩn.

In [ ]:
import base64
import os
import platform
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

token_val = userdata.get("HF_TOKEN")
if not token_val:
    raise RuntimeError("Missing HF_TOKEN in Colab Secrets.")
os.environ["HF_TOKEN"] = token_val.strip()
del token_val

REPO_URL = "https://github.com/thienandinh29/financial-sentiment-dl.git"
REPO_DIR = Path("financial-sentiment-dl")
BRANCH = "tv3/week2"

if not REPO_DIR.exists():
    gh_val = userdata.get("GH_TOKEN")
    if not gh_val:
        raise RuntimeError("Missing GH_TOKEN in Colab Secrets.")
    basic_auth = base64.b64encode(f"x-access-token:{gh_val.strip()}".encode("utf-8")).decode("ascii")
    git_env = os.environ.copy()
    git_env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic_auth}",
    })
    del gh_val, basic_auth
    res = subprocess.run(
        ["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        env=git_env,
        capture_output=True,
        text=True,
        check=False,
    )
    del git_env
    if res.returncode != 0:
        raise RuntimeError(f"git clone failed with exit code {res.returncode}.")

if Path.cwd().name != REPO_DIR.name and REPO_DIR.exists():
    os.chdir(REPO_DIR)

remote_origin = subprocess.run(
    ["git", "config", "--get", "remote.origin.url"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
assert remote_origin == REPO_URL, "Unexpected remote.origin.url format."

py_ver = platform.python_version()
print(f"Python version: {py_ver}")
if Path("requirements.lock").is_file():
    lock_proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.lock"],
        capture_output=True,
        text=True,
        check=False,
    )
    if lock_proc.returncode == 0:
        print("Installed dependencies from requirements.lock successfully.")
    else:
        print(f"requirements.lock failed on Python {py_ver}; falling back to requirements.txt.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not (Path("data/raw/fns/train.jsonl").is_file() and Path("data/raw/fns/validation.jsonl").is_file()):
    subprocess.run([sys.executable, "data/download.py", "--only", "fns"], check=True)

## 2. Nạp Mô hình `Qwen/Qwen2.5-7B-Instruct` 4-bit NF4 & Hàm `letter_probs(prompt)`

- Kiểm tra bắt buộc CUDA GPU trước khi nạp mô hình.
- Hàm `letter_probs(prompt)` nhận toàn bộ chuỗi prompt đã định dạng (`P1`, `P2` hoặc `P3`), lấy `logits[0, -1]`, gộp xác suất của biến thể có/không có khoảng trắng (`"A"` và `" A"`) bằng `torch.logsumexp`, và chuẩn hóa `softmax` theo đúng thứ tự `bearish`, `neutral`, `bullish`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not (torch.cuda.is_available() and torch.cuda.device_count() > 0):
    raise RuntimeError(
        "CUDA GPU is required to load Qwen/Qwen2.5-7B-Instruct in 4-bit NF4 (bitsandbytes). "
        "Please enable a GPU runtime in Google Colab."
    )

SEED = 1
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

name = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
model.eval()

LETTERS = {"A": "bearish", "B": "neutral", "C": "bullish"}
LETTER_IDS = {
    L: sorted({ids[0] for v in (L, " " + L) if len(ids := tok.encode(v, add_special_tokens=False)) == 1})
    for L in LETTERS
}
for letter, ids in LETTER_IDS.items():
    if not ids:
        raise ValueError(f"No single-token IDs found for option {letter!r}.")
all_letter_ids = [tid for ids in LETTER_IDS.values() for tid in ids]
assert len(all_letter_ids) == len(set(all_letter_ids)), f"Overlapping token IDs: {LETTER_IDS}"


def letter_probs(prompt: str):
    """Return (probs over bearish/neutral/bullish, mass on A/B/C tokens, top-10 next tokens)."""
    chat = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tok(chat, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    logp = torch.log_softmax(logits, dim=-1)
    per_letter = torch.stack([torch.logsumexp(logp[LETTER_IDS[L]], dim=0) for L in LETTERS])
    probs = torch.softmax(per_letter, dim=0)
    top10 = [tok.decode([i]) for i in logits.topk(10).indices.tolist()]
    return dict(zip(LETTERS.values(), probs.tolist())), per_letter.exp().sum().item(), top10

print("Loaded model on GPU:", torch.cuda.get_device_name(0), "| LETTER_IDS:", LETTER_IDS)

## 3. Định nghĩa Ba Prompt `P1`, `P2`, `P3` & Các Hàm Đánh giá (`docs/guides/TV3-tuan2.md`)

- `P1`: Prompt ngắn gọn từ Tuần 1.
- `P2`: Prompt kèm định nghĩa 3 nhãn theo `labeling/GUIDE.md`.
- `P3`: Prompt few-shot lấy 1 ví dụ mỗi nhãn từ tập `train` (`seed=1`), **không bao giờ lấy từ `validation` hay `test`**.

In [ ]:
import json
import os
import random
from pathlib import Path

PROMPTS = {
    "P1": (
        "Classify the sentiment of this financial news headline for the main company or market it is about.\n"
        "Headline: {text}\n"
        "A) bearish\nB) neutral\nC) bullish\n"
        "Answer with one letter only."
    ),
    "P2": (
        "You label financial news headlines.\n"
        "bearish = a clearly unfavourable development or outlook for the main subject "
        "(price falls, losses, downgrades, fines).\n"
        "bullish = a clearly favourable development or outlook for the main subject "
        "(price rises, beats estimates, upgrades).\n"
        "neutral = financial information with no clear direction.\n"
        "Judge only what the headline says.\n"
        "Headline: {text}\n"
        "A) bearish\nB) neutral\nC) bullish\n"
        "Answer with one letter only."
    ),
    "P3": (
        "Classify the sentiment of a financial news headline for its main subject.\n"
        "{examples}"
        "Headline: {text}\n"
        "A) bearish\nB) neutral\nC) bullish\n"
        "Answer with one letter only."
    ),
}
LETTER_OF = {0: "A", 1: "B", 2: "C"}          # A/B/C = LABELS order: bearish, neutral, bullish


def few_shot_block(train_examples, per_label=1, seed=1):
    """Pick one train example per label (seeded) and format them for P3; train only, never validation."""
    rng = random.Random(seed)
    lines = []
    text_by_id = {i: t for i, t, _ in train_examples}
    for label in (0, 1, 2):
        pool = sorted(i for i, _, y in train_examples if y == label)
        chosen = rng.sample(pool, per_label)
        lines += [f"Headline: {text_by_id[i]}\nAnswer: {LETTER_OF[label]}\n" for i in chosen]
    return "Examples:\n" + "".join(lines) + "\n"


def run_prompt(name, val_examples, letter_probs_fn, examples=""):
    """Score every validation headline with one prompt; return (preds dict, check dict without text)."""
    preds, masses, top1_is_letter = {}, [], 0
    for i, text, _ in val_examples:
        probs, mass, top10 = letter_probs_fn(PROMPTS[name].format(text=text, examples=examples))
        preds[i] = [probs["bearish"], probs["neutral"], probs["bullish"]]
        masses.append(mass)
        top1_is_letter += top10[0].strip() in ("A", "B", "C")
    check = {
        "prompt": name,
        "n": len(val_examples),
        "min_abc_mass": round(min(masses), 4),
        "mean_abc_mass": round(sum(masses) / len(masses), 4),
        "share_top1_is_letter": round(top1_is_letter / len(val_examples), 4),
    }
    return preds, check


def write_preds(path, preds):
    """Write {"id", "probs"} lines (shared contract)."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for i, p in preds.items():
            f.write(json.dumps({"id": i, "probs": [round(float(x), 6) for x in p]}) + "\n")

## 4. Chạy Đánh giá `P1`, `P2`, `P3` trên Tập Validation & Xuất `top10_check.json`

- Đọc `data/splits/cluster_split.jsonl` với mã băm SHA-256 `8c2c35ea789acf5b2bd9c74c5e59fa0df325034422be42ae7b47f2e8c9c46aaa`.
- Lưu dự đoán của từng prompt vào `results/E11/{P1,P2,P3}/preds_validation.jsonl`.
- Chấm điểm qua `evaluate.metrics.score()` và lưu bảng tổng hợp (không chứa văn bản tiêu đề) vào `results/llm_prompts/top10_check.json`.

In [ ]:
import json
import os
import time
from evaluate.metrics import read_preds, score
from train.data import load_split_data

SPLIT = "data/splits/cluster_split.jsonl"
SHA = "8c2c35ea789acf5b2bd9c74c5e59fa0df325034422be42ae7b47f2e8c9c46aaa"

data = load_split_data(SPLIT, SHA)
val = data["validation"]
gold = {i: y for i, _, y in val}
examples = few_shot_block(data["train"], per_label=1, seed=1)

checks = []
for prompt_name in PROMPTS:                                     # ~10 minutes per prompt on Tesla T4
    t0 = time.time()
    preds, check = run_prompt(prompt_name, val, letter_probs, examples=examples)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_s = round(time.time() - t0, 2)
    path = f"results/E11/{prompt_name}/preds_validation.jsonl"
    write_preds(path, preds)
    check.update(score(read_preds(path), gold))
    check["elapsed_seconds"] = elapsed_s
    checks.append(check)
    print(prompt_name, check["macro_f1"], check["min_abc_mass"], check["share_top1_is_letter"], f"{elapsed_s}s")

os.makedirs("results/llm_prompts", exist_ok=True)
with open("results/llm_prompts/top10_check.json", "w", encoding="utf-8") as f:
    json.dump(checks, f, indent=2)